In [ ]:
 !pip install librosa moviepy mediapipe tqdm scikit-learn -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d ejlok1/cremad -p /content/CREMAD --unzip

In [ ]:
import os
import pandas as pd
import glob

DATA_PATH = "/content/CREMAD/AudioWAV"
files_list = glob.glob(os.path.join(DATA_PATH, "*.wav"))

In [ ]:
print(f"Total audio files: {len(files_list)}")

In [ ]:
emotion_map = {
    'ANG': 'angry', 'DIS': 'disgust', 'FEA': 'fear',
    'HAP': 'happy',  'NEU': 'neutral', 'SAD': 'sad'
}

In [ ]:
records = []
for f in files_list:
    parts = os.path.basename(f).replace('.wav','').split('_')
    if len(parts) >= 3:
        emotion_code = parts[2]
        label = emotion_map.get(emotion_code, None)
        if label:
            records.append({'path': f, 'emotion': label, 'code': emotion_code})

df = pd.DataFrame(records)
print(f"Total samples: {len(df)}")
print(df['emotion'].value_counts())

In [ ]:
import os
import pandas as pd
import glob
import librosa
import numpy as np
from tqdm import tqdm

# Code from KQ0O2w8rvL4Y
DATA_PATH = "/content/CREMAD/AudioWAV"
files_list = glob.glob(os.path.join(DATA_PATH, "*.wav"))

# Code from xSxhZmHevuPX
emotion_map = {
    'ANG': 'angry', 'DIS': 'disgust', 'FEA': 'fear',
    'HAP': 'happy',  'NEU': 'neutral', 'SAD': 'sad'
}

# Code from JzjexCkFvulI
records = []
for f in files_list:
    parts = os.path.basename(f).replace('.wav','').split('_')
    if len(parts) >= 3:
        emotion_code = parts[2]
        label = emotion_map.get(emotion_code, None)
        if label:
            records.append({'path': f, 'emotion': label, 'code': emotion_code})
df = pd.DataFrame(records)

def extract_audio_features(file_path, max_pad=174):
    try:
        y, sr = librosa.load(file_path, duration=3.0, offset=0.5)

        def pad(feat):
            if feat.shape[1] < max_pad:
                feat = np.pad(feat, ((0,0),(0, max_pad - feat.shape[1])), mode='constant')
            return feat[:, :max_pad]

        mfcc     = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        mfcc_d   = librosa.feature.delta(mfcc)
        mfcc_d2  = librosa.feature.delta(mfcc, order=2)
        mel      = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=40)
        mel      = librosa.power_to_db(mel, ref=np.max)
        chroma   = librosa.feature.chroma_stft(y=y, sr=sr, n_chroma=12)
        zcr      = librosa.feature.zero_crossing_rate(y)
        rms      = librosa.feature.rms(y=y)
        spec_bw  = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        rolloff  = librosa.feature.spectral_rolloff(y=y, sr=sr)

        features = np.vstack([
            pad(mfcc), pad(mfcc_d), pad(mfcc_d2),
            pad(mel), pad(chroma),
            pad(zcr), pad(rms), pad(spec_bw), pad(rolloff),
        ])  # (176, 174)

        return features
    except Exception as e:
        print(f"Error extracting features from {file_path}: {e}")
        return None

print("Extracting audio features...")
audio_features, labels, valid_paths = [], [], []

for _, row in tqdm(df.iterrows(), total=len(df)):
    feat = extract_audio_features(row['path'])
    if feat is not None:
        audio_features.append(feat)
        labels.append(row['emotion'])
        valid_paths.append(row['path'])

audio_features = np.array(audio_features)
print(f"Feature shape: {audio_features.shape}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TemporalAttention(nn.Module):
    """Learns WHICH timesteps matter most for emotion"""
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, 1)

    def forward(self, lstm_out):
        # lstm_out: (B, T, H)
        weights = torch.softmax(self.attn(lstm_out), dim=1)  # (B, T, 1)
        context = (weights * lstm_out).sum(dim=1)             # (B, H)
        return context

class AudioCNNLSTM(nn.Module):
    def __init__(self, num_classes=6, input_channels=176):
        super().__init__()
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=(3,3), padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2,2), nn.Dropout2d(0.1),
            # Block 2
            nn.Conv2d(32, 64, kernel_size=(3,3), padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2,2), nn.Dropout2d(0.1),
            # Block 3
            nn.Conv2d(64, 128, kernel_size=(3,3), padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 22)),  # fixed output size
        )
        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=128, # Corrected from 128 * 4
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            dropout=0.35,
            bidirectional=True
        )
        self.attention = TemporalAttention(hidden_size=256)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)             # (B, 128, 4, 22)
        x = x.flatten(2)            # (B, 128, 88)
        x = x.permute(0, 2, 1)     # (B, 88, 128)
        lstm_out, _ = self.lstm(x)  # (B, 88, 512) - hidden_size * 2 for bidirectional
        context = self.attention(lstm_out)  # (B, 512) — weighted sum
        return self.classifier(context)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AudioCNNLSTM(num_classes=6).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Device: {device} | Parameters: {total_params:,}")

In [ ]:
import random
from torch.utils.data import Dataset

class AudioDatasetAug(Dataset):
    """Training dataset with on-the-fly augmentation"""
    def __init__(self, features, labels, augment=True):
        self.X = features
        self.y = torch.LongTensor(labels)
        self.augment = augment

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        x = self.X[i].copy()  # (176, 174)

        if self.augment and random.random() < 0.5:
            t_start = random.randint(0, 130)
            t_len   = random.randint(10, 30)
            x[:, t_start:t_start+t_len] = 0

        if self.augment and random.random() < 0.4:
            f_start = random.randint(0, 140)
            f_len   = random.randint(5, 20)
            x[f_start:f_start+f_len, :] = 0

        if self.augment and random.random() < 0.3:
            x += np.random.normal(0, 0.01, x.shape)

        x = torch.FloatTensor(x).unsqueeze(0)  # (1, 176, 174)
        return x, self.y[i]

class AudioDataset(Dataset):
    """Eval dataset — no augmentation"""
    def __init__(self, features, labels):
        self.X = torch.FloatTensor(features).unsqueeze(1)
        self.y = torch.LongTensor(labels)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

le = LabelEncoder()
y_encoded = le.fit_transform(labels)
print("Classes:", le.classes_)

# Train/Val/Test split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    audio_features, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# ✅ NEW: Normalize AFTER split, fit only on train
n_samples_tr, n_feat, n_time = X_train.shape
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train.reshape(n_samples_tr, -1)).reshape(n_samples_tr, n_feat, n_time)
X_val   = scaler.transform(X_val.reshape(len(X_val), -1)).reshape(len(X_val), n_feat, n_time)
X_test  = scaler.transform(X_test.reshape(len(X_test), -1)).reshape(len(X_test), n_feat, n_time)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
import torch
from torch.utils.data import DataLoader
# ✅ FIX: use AudioDatasetAug for training, not AudioDataset
train_loader = DataLoader(AudioDatasetAug(X_train, y_train, augment=True), batch_size=32, shuffle=True)
val_loader   = DataLoader(AudioDataset(X_val, y_val), batch_size=32)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# ✅ FIX: CosineAnnealingLR is smoother and works better than ReduceLROnPlateau here
EPOCHS = 80
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# ✅ FIX: label_smoothing=0.1 reduces overconfidence on 6 classes
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def train_epoch(model, loader):
    model.train(); total_loss, correct = 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward(); optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == y).sum().item()
    return total_loss/len(loader), correct/len(loader.dataset)

def eval_epoch(model, loader):
    model.eval(); total_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            total_loss += criterion(out, y).item()
            correct += (out.argmax(1) == y).sum().item()
    return total_loss/len(loader), correct/len(loader.dataset)

best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = train_epoch(model, train_loader)
    val_loss, val_acc = eval_epoch(model, val_loader)

    # ✅ FIX: step scheduler every epoch, not on val_loss
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_audio_model.pt')

    if epoch % 10 == 0 or epoch == 1:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:02d} | Train: {tr_acc:.3f} | Val: {val_acc:.3f} | LR: {lr:.2e}")

print(f"\nBest Val Accuracy: {best_val_acc:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model.load_state_dict(torch.load('best_audio_model.pt'))
test_loader = DataLoader(AudioDataset(X_test, y_test), batch_size=32)

all_preds, all_true = [], []
model.eval()
with torch.no_grad():
    for X, y in test_loader:
        out = model(X.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_true.extend(y.numpy())

print(classification_report(all_true, all_preds, target_names=le.classes_))

cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.title('Confusion Matrix — Audio Model')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, accuracy_score
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Load best model and run on test set
model.load_state_dict(torch.load('best_audio_model.pt'))
model.eval()

test_loader = DataLoader(AudioDataset(X_test, y_test), batch_size=64)

all_preds, all_true, all_probs = [], [], []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        out = model(X)
        probs = torch.softmax(out, dim=1).cpu().numpy()
        preds = out.argmax(1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_true.extend(y.numpy())

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_true  = np.array(all_true)

# Summary metrics
acc  = accuracy_score(all_true, all_preds)
f1_w = f1_score(all_true, all_preds, average='weighted')
f1_m = f1_score(all_true, all_preds, average='macro')

print("=" * 50)
print(f"  Test Accuracy        : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Weighted F1-Score    : {f1_w:.4f}")
print(f"  Macro F1-Score       : {f1_m:.4f}")
print("=" * 50)
print("\nPer-class Report:\n")
print(classification_report(all_true, all_preds, target_names=le.classes_))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,14))
fig.suptitle('Training Overview', fontsize=15, fontweight='bold', y=1.02)

epochs_range = range(1, len(history['train_acc']) + 1)

# Accuracy
axes[0].plot(epochs_range, history['train_acc'], color='#378ADD', linewidth=2, label='Train')
axes[0].plot(epochs_range, history['val_acc'],   color='#1D9E75', linewidth=2, label='Validation')
axes[0].axhline(y=best_val_acc, color='#D85A30', linestyle='--', linewidth=1.5,
                label=f'Best Val: {best_val_acc:.3f}')
axes[0].fill_between(epochs_range, history['train_acc'], history['val_acc'],
                     alpha=0.08, color='#888780')
axes[0].set_title('Accuracy over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1)

# Highlight best epoch
best_ep = np.argmax(history['val_acc']) + 1
axes[0].axvline(x=best_ep, color='#1D9E75', linestyle=':', alpha=0.7)
axes[0].annotate(f'Best\nEp {best_ep}', xy=(best_ep, best_val_acc),
                 xytext=(best_ep+2, best_val_acc-0.08),
                 fontsize=9, color='#1D9E75',
                 arrowprops=dict(arrowstyle='->', color='#1D9E75'))

# Gap (overfitting check)
gap = [t - v for t, v in zip(history['train_acc'], history['val_acc'])]
axes[1].plot(epochs_range, gap, color='#D85A30', linewidth=2)
axes[1].fill_between(epochs_range, gap, 0,
                     where=[g > 0 for g in gap], alpha=0.15, color='#D85A30', label='Overfit zone')
axes[1].fill_between(epochs_range, gap, 0,
                     where=[g <= 0 for g in gap], alpha=0.15, color='#378ADD', label='Underfit zone')
axes[1].axhline(y=0, color='gray', linewidth=1)
axes[1].set_title('Train-Val Gap (Overfitting Check)', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Train Acc - Val Acc')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrix', fontsize=15, fontweight='bold')

cm      = confusion_matrix(all_true, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

emotion_labels = le.classes_

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0],
            xticklabels=emotion_labels, yticklabels=emotion_labels,
            cmap='Blues', linewidths=0.5, linecolor='white',
            annot_kws={'size': 11})
axes[0].set_title('Raw Counts', fontweight='bold')
axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')
axes[0].tick_params(axis='x', rotation=30)

# Normalized (shows class-wise accuracy clearly)
sns.heatmap(cm_norm, annot=True, fmt='.2f', ax=axes[1],
            xticklabels=emotion_labels, yticklabels=emotion_labels,
            cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 11})
axes[1].set_title('Normalized (Recall per Class)', fontweight='bold')
axes[1].set_ylabel('True Label'); axes[1].set_xlabel('Predicted Label')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    all_true, all_preds, labels=range(len(le.classes_)))

x     = np.arange(len(le.classes_))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 5))

bars1 = ax.bar(x - width,     precision, width, label='Precision', color='#378ADD', alpha=0.85)
bars2 = ax.bar(x,             recall,    width, label='Recall',    color='#1D9E75', alpha=0.85)
bars3 = ax.bar(x + width,     f1,        width, label='F1-Score',  color='#D85A30', alpha=0.85)

# Value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.2f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels([e.capitalize() for e in le.classes_], fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score'); ax.set_title('Per-Class Metrics', fontweight='bold', fontsize=13)
ax.axhline(y=acc, color='purple', linestyle='--', alpha=0.5, linewidth=1.5,
           label=f'Overall Acc: {acc:.3f}')
ax.legend(loc='upper right'); ax.grid(axis='y', alpha=0.3)

# Sample count on x-axis
for i, (e, s) in enumerate(zip(le.classes_, support)):
    ax.text(i, -0.08, f'n={s}', ha='center', fontsize=8.5, color='gray',
            transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.savefig('per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_bin = label_binarize(all_true, classes=range(len(le.classes_)))

colors = ['#378ADD','#1D9E75','#D85A30','#7F77DD','#BA7517','#D4537E']
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

roc_aucs = []
for i, (cls, color) in enumerate(zip(le.classes_, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    roc_aucs.append(roc_auc)

    axes[i].plot(fpr, tpr, color=color, lw=2.5, label=f'AUC = {roc_auc:.3f}')
    axes[i].plot([0,1],[0,1], 'k--', lw=1, alpha=0.5)
    axes[i].fill_between(fpr, tpr, alpha=0.08, color=color)
    axes[i].set_xlim([0,1]); axes[i].set_ylim([0,1.02])
    axes[i].set_title(f'{cls.capitalize()}', fontweight='bold', fontsize=12)
    axes[i].set_xlabel('FPR'); axes[i].set_ylabel('TPR')
    axes[i].legend(loc='lower right', fontsize=10)
    axes[i].grid(True, alpha=0.25)

mean_auc = np.mean(roc_aucs)
fig.suptitle(f'ROC Curves — One vs Rest  |  Mean AUC: {mean_auc:.3f}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
colors = ['#378ADD','#1D9E75','#D85A30','#7F77DD','#BA7517','#D4537E']

for i, (cls, color) in enumerate(zip(le.classes_, colors)):
    mask    = all_true == i
    correct = all_preds[mask] == all_true[mask]
    conf    = all_probs[mask, i]

    axes[i].hist(conf[correct],  bins=20, alpha=0.7, color=color,   label='Correct')
    axes[i].hist(conf[~correct], bins=20, alpha=0.5, color='#E24B4A', label='Wrong')
    axes[i].axvline(conf.mean(), color='black', linestyle='--', linewidth=1.5,
                    label=f'Mean: {conf.mean():.2f}')
    axes[i].set_title(f'{cls.capitalize()}  (acc: {correct.mean():.2f})',
                      fontweight='bold', fontsize=11)
    axes[i].set_xlabel('Predicted Confidence')
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=8.5)
    axes[i].grid(True, alpha=0.25)

fig.suptitle('Prediction Confidence Distribution per Emotion',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confidence_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:


import IPython.display as ipd
import librosa.display

def predict_single(file_path, true_label=None):
    """Full pipeline for one audio file — plays audio + shows spectrogram + prediction"""

    y_audio, sr = librosa.load(file_path, duration=3.0, offset=0.5)
    feat = extract_audio_features(file_path)

    if feat is None:
        print("Feature extraction failed."); return

    x = torch.FloatTensor(feat).unsqueeze(0).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        out   = model(x)
        probs = torch.softmax(out, dim=1).cpu().numpy()[0]
        pred  = le.classes_[probs.argmax()]

    # --- Plot ---
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Waveform
    librosa.display.waveshow(y_audio, sr=sr, ax=axes[0], color='#378ADD')
    axes[0].set_title('Waveform', fontweight='bold')
    axes[0].set_xlabel('Time (s)')

    # Mel Spectrogram
    mel = librosa.feature.melspectrogram(y=y_audio, sr=sr, n_mels=40)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr, x_axis='time',
                                   y_axis='mel', ax=axes[1], cmap='viridis')
    fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
    axes[1].set_title('Mel Spectrogram', fontweight='bold')

    # Prediction bars
    colors_bar = ['#1D9E75' if c == pred else '#B4B2A9' for c in le.classes_]
    bars = axes[2].barh(le.classes_, probs * 100, color=colors_bar, edgecolor='white')
    for bar, p in zip(bars, probs):
        axes[2].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                     f'{p*100:.1f}%', va='center', fontsize=9.5)
    axes[2].set_xlim(0, 115)
    axes[2].set_xlabel('Confidence (%)')
    axes[2].set_title('Prediction', fontweight='bold')
    axes[2].grid(axis='x', alpha=0.3)

    title_color = '#1D9E75' if (true_label is None or pred == true_label) else '#E24B4A'
    status = '' if true_label is None else (' ✓' if pred == true_label else f' ✗ (true: {true_label})')
    fig.suptitle(f'Predicted: {pred.upper()}{status}',
                 fontsize=14, fontweight='bold', color=title_color)

    plt.tight_layout()
    plt.savefig('single_prediction.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Play the audio in Colab
    display(ipd.Audio(file_path))
    return pred, probs

# --- Run on a random test sample ---
sample_idx   = np.random.randint(len(X_test))
sample_path  = valid_paths[len(X_train) + len(X_val) + sample_idx]  # approx mapping
true_emotion = le.classes_[y_test[sample_idx]]

pred, probs = predict_single(sample_path, true_label=true_emotion)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    all_true, all_preds, labels=range(len(le.classes_)))

mean_auc = np.mean(roc_aucs)

fig = plt.figure(figsize=(10, 4))
fig.patch.set_facecolor('#F8F9FA')

best_ep = np.argmax(history['val_acc']) + 1

summary = {
    'Test Accuracy':    f'{acc*100:.2f}%',
    'Weighted F1':      f'{f1_w:.4f}',
    'Macro F1':         f'{f1_m:.4f}',
    'Mean ROC-AUC':     f'{mean_auc:.4f}',
    'Best Val Epoch':   f'{best_ep}',
    'Test Samples':     f'{len(all_true)}',
}

colors_card = ['#E6F1FB','#E1F5EE','#FAECE7','#EEEDFE','#FAEEDA','#EAF3DE']

for i, (k, v) in enumerate(summary.items()):
    ax = fig.add_axes([0.03 + (i % 3) * 0.33, 0.55 - (i // 3) * 0.52, 0.29, 0.42])
    ax.set_facecolor(colors_card[i])
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.text(0.5, 0.65, v,  ha='center', va='center', fontsize=20, fontweight='bold',
            transform=ax.transAxes, color='#1a1a2e')
    ax.text(0.5, 0.22, k, ha='center', va='center', fontsize=10,
            transform=ax.transAxes, color='#555')

fig.suptitle('Model Performance Summary', fontsize=14,
             fontweight='bold', y=1.02, color='#1a1a2e')
plt.savefig('summary_card.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import joblib
import os

os.makedirs('saved_model', exist_ok=True)

torch.save(model.state_dict(), 'saved_model/audio_model.pt')
joblib.dump(scaler, 'saved_model/scaler.pkl')
joblib.dump(le, 'saved_model/label_encoder.pkl')

print("Done. Files saved:")
for f in os.listdir('saved_model'):
    size = os.path.getsize(f'saved_model/{f}') / 1024
    print(f"  {f}  ({size:.1f} KB)")


In [ ]:
import h5py

# Path for the H5 file
h5_path = 'saved_model/audio_model_full.h5'

# Save weights to H5
state_dict = model.state_dict()
with h5py.File(h5_path, 'w') as f:
    # Create a group for weights
    weight_group = f.create_group('weights')
    for key, value in state_dict.items():
        weight_group.create_dataset(key, data=value.cpu().numpy())

    # Store metadata like classes
    f.attrs['classes'] = [c.encode('utf-8') for c in le.classes_]
    print(f"Model weights and metadata saved to {h5_path}")